# EDA — Advanced SIEM Dataset

**Dataset:** [`darkknight25/Advanced_SIEM_Dataset`](https://huggingface.co/datasets/darkknight25/Advanced_SIEM_Dataset)

100 000 eventos de seguridad **sintéticos** que imitan logs de un SIEM (firewall, IDS, autenticación, endpoint, red, nube, IoT, IA).

**Formato:** JSON Lines. Campos comunes (`event_id`, `timestamp`, `event_type`, `severity`, `raw_log`…) + campos que dependen del tipo de evento + dos objetos anidados (`advanced_metadata`, `behavioral_analytics`).

**Preguntas guía para el EDA**
- ¿Cómo se distribuyen tipos de evento y severidades? ¿Es realista o demasiado uniforme?
- ¿Qué columnas aplican a cada tipo de evento?
- ¿Cómo se ve la línea de tiempo?
- ¿`risk_score` tiene relación con `severity`?

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Busca la raíz del repo (donde está src/profiler.py) y la agrega al path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "profiler.py").exists())
sys.path.insert(0, str(ROOT))
from src.profiler import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

ctx = eda_context("09")   # rutas: datos en data/raw/..., resultados en esta carpeta
print(ctx)

## 1. Inventario de archivos

In [ ]:
files = list_files(ctx.data_dir)
display(summarize_extensions(files))
display(files)

## 2. Cargar (100k filas, cabe en memoria) y aplanar los objetos anidados

In [ ]:
PATH = ctx.data_dir / "advanced_siem_dataset.jsonl"
raw = read_jsonl(PATH)
nested = [c for c in ["advanced_metadata", "behavioral_analytics"] if c in raw.columns]
flat = [pd.json_normalize(raw[c].apply(lambda x: x if isinstance(x, dict) else {})).add_prefix(f"{c}.") for c in nested]
df = pd.concat([raw.drop(columns=nested)] + flat, axis=1)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
print(df.shape)
display(df.head(3))
display(profile_df(df))

## 3. Tipos de evento y severidad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
df["event_type"].value_counts().plot.bar(ax=axes[0]); axes[0].set_title("event_type")
df["severity"].value_counts().plot.bar(ax=axes[1]); axes[1].set_title("severity")
sns.heatmap(pd.crosstab(df["event_type"], df["severity"], normalize="index"), annot=True, fmt=".0%", cmap="Oranges", ax=axes[2])
axes[2].set_title("Severidad dentro de cada tipo")
plt.tight_layout(); save_fig(fig, ctx, "tipos_y_severidad"); plt.show()

## 4. Columnas por tipo de evento
Muchas columnas solo aplican a un tipo de evento; el mapa muestra el % de valores no nulos.

In [ ]:
fill = df.notna().groupby(df["event_type"]).mean().T
fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(fill, cmap="Greens", ax=ax, cbar_kws={"label": "% no nulo"})
ax.set_title("Cobertura de columnas por event_type"); save_fig(fig, ctx, "cobertura_columnas"); plt.show()

## 5. Tiempo y puntajes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
df.set_index("timestamp").resample("W").size().plot(ax=axes[0]); axes[0].set_title("Eventos por semana")
if "advanced_metadata.risk_score" in df:
    sns.boxplot(data=df, x="severity", y="advanced_metadata.risk_score", ax=axes[1],
                order=["info", "low", "medium", "high", "critical", "emergency"])
    axes[1].set_title("risk_score por severidad")
plt.tight_layout(); save_fig(fig, ctx, "tiempo_y_riesgo"); plt.show()
print("Rango de fechas:", df["timestamp"].min(), "→", df["timestamp"].max())

In [ ]:
for c in ["source", "action", "advanced_metadata.geo_location", "alert_type", "category", "protocol"]:
    if c in df:
        print(f"\n--- {c} (top 10 de {df[c].nunique()}) ---")
        display(df[c].value_counts().head(10).to_frame("n"))

## 6. Ejemplos de logs crudos

In [ ]:
show_examples(df, n=4, cols=["event_type", "severity", "raw_log"], max_chars=300)

## Guardar resultados
Se guardan en `tables/` de esta carpeta. `resumen.csv` lo consolida `eda/00_general`.

In [ ]:
save_table(profile_df(df), ctx, "perfil_columnas")
display(save_summary(ctx, "darkknight25/Advanced_SIEM_Dataset", df, total_rows=None, dup_cols=["event_id"],
                     notas=""))

## Próximos pasos / notas
- Anota tus hallazgos en el `README.md` de esta carpeta.
-